# Kandidat ukuran lebih kecil: satu bank palette 6-bit per encoder

Audit menunjukkan hampir seluruh paket adalah blob bobot, tidak ditemukan encoding/layout berbeda yang decode-nya identik, dan pengemasan sebelumnya telah menghapus kira-kira jumlah duplikasi identik yang terukur. Mengulang pengemasan lossless yang sama tidak memberikan pengurangan besar tambahan.

Kandidat ini memakai **bank palette dari `long_120` yang sudah ada** sebagai acuan tetap untuk `short_30`, `long_120`, dan `partial`. Tidak menjalankan training atau KMeans/kuantisasi ulang. **Nilai bobot terkuantisasi short/partial berubah**, sehingga hasil kualitas lama tidak menjadi bukti kualitas kandidat ini. Kita memilih acuan sebelum evaluasi, tanpa mencari beberapa acuan berdasarkan hasil validation.

Nama parameter, dtype 6-bit/fp32 dan layout harus cocok. Hanya penambahan/pengurangan singleton trailing dimension yang didukung; tidak menebak transpose atau nama parameter. Sanity guard menolak perbedaan RMS parameter relatif di atas 10% sebagai dugaan binding yang tidak cocok; ini bukan ukuran keberhasilan akurasi. Operator jaringan selain konstanta parameter harus tetap sama. Profil acuan 120 detik harus mempertahankan seluruh hash graph dan lulus kontrol native.

Ekspor dan pengemasan tetap di `outputs.local`, dengan sumber tidak ditimpa. MusicFM/MuQ dijalankan dalam proses terpisah. Setelah hasil compiled benar-benar lebih kecil, lakukan smoke satu lagu lalu **32 lagu / seluruh 451 target manual**, termasuk menghitung ulang partial window dengan bobot baru. Tidak memakai embedding partial lama sebagai fitur kandidat. Head/konteks epoch 5, boundary, anotasi, split dan test tetap.

Kandidat hanya bisa dipertimbangkan bila akurasi, macro F1 dan F1 setiap kelas tidak turun. GPU iPhone, latency, memori dan ukuran aplikasi final belum diuji untuk paket baru ini. Tidak ada pemasangan otomatis ke SongMap.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib,json,subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/canonical_encoder_weights.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'  # Jangan resolve symlink executable.
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED_PACK={'musicfm':'1f5abdd9aa4fa4d3dec4cf0cc038a203a4aae8e914218876a270c668da500f5d',
               'muq':'3c5a5bf912f393cbdef908af69642f9899ea06f06a4bcfba18b44770e39f8149'}
EXPECTED_AUDIT={'musicfm':'6b2d61183c9ac061eba271e29571793097d0d44f94e59f25a6f70e5c0c19f5e2',
                'muq':'290cb5311d89eb6a44085cbf04e6f5649b11373e3ab70f189d2a67db08a227c2'}
PACK={};AUDIT={}
for role in EXPECTED_PACK:
    PACK[role]=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-weight-packaging-*/'+role+'-*/packaging-report.json') if sha(p)==EXPECTED_PACK[role]),None)
    AUDIT[role]=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-sharing-audit-*/'+role+'-audit.json') if sha(p)==EXPECTED_AUDIT[role]),None)
    assert PACK[role] and AUDIT[role], f'Report {role} yang sesuai belum ditemukan'
QUALITY=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-glu-quality-*/validation-all.json') if sha(p)=='3170c7da201b3f1df8866d89862dcdeef9e08ff47d96206a07f3d21e39da5358'),None)
REPAIR=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-glu-repair-*/repair-report.json') if sha(p)=='3e84fe326e03a8f0e0eed1bff07c1fe00ea15cbf7ed3175f70c8fa9e14e408c0'),None)
assert QUALITY and REPAIR, 'Report sumber kualitas/repair belum ditemukan'
selection=json.loads((QUALITY.parent/'selection.json').read_text())
PALETTE_VALIDATION=next((p.resolve() for p in (ROOT/'outputs.local').glob('context-palette6-validation-*/validation-all.json') if sha(p)==selection['context_identity']['palette_validation_sha256']),None)
assert PALETTE_VALIDATION
palette_selection=json.loads((PALETTE_VALIDATION.parent/'selection.json').read_text())
FROZEN=next((p.parent.resolve() for p in (ROOT/'outputs.local').glob('frozen-context-test-*/selection.json') if json.loads(p.read_text()).get('identity')==palette_selection['frozen_selection_identity']),None)
assert FROZEN, 'Snapshot konteks tidak ditemukan'
BASELINE_POOL=ROOT/'outputs.local/section-context-palette6-cache-v1'
RESUME_OUTPUT=None  # Isi folder encoder-canonical-bank-* ini setelah restart kernel jika ingin melanjutkan.
OUTPUT=(Path(RESUME_OUTPUT).expanduser().resolve() if RESUME_OUTPUT else ROOT/'outputs.local'/('encoder-canonical-bank-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')))
REPORTS={}
print('Output:',OUTPUT)

def run_logged(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen([str(v) for v in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Proses gagal ({status}); lihat {log}. Model sumber dan hasil yang sudah selesai tetap tersimpan.')

def export(role):
    done=[]
    for path in OUTPUT.glob(role+'-*/canonical-report.json'):
        r=json.loads(path.read_text())
        if (r.get('candidate_for_quality_validation') and not r.get('error')
                and r.get('script_sha256')==sha(ROOT/'scripts/canonical_encoder_weights.py')
                and r.get('audit_sha256')==EXPECTED_AUDIT[role] and r.get('prior_packaging_sha256')==EXPECTED_PACK[role]):done.append(path)
    if done:
        REPORTS[role]=max(done,key=lambda p:p.stat().st_mtime);print('Ekspor selesai digunakan kembali:',REPORTS[role]);return
    attempt=OUTPUT/(role+'-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
    run_logged([RUN_PYTHON,'-u',ROOT/'scripts/canonical_encoder_weights.py',
        '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,'--quality-report',QUALITY,'--graph-repair',REPAIR,
        '--packaging-report',PACK[role],'--audit-report',AUDIT[role],'--encoder',role,'--output-dir',attempt],Path(str(attempt)+'.export.log'))
    REPORTS[role]=attempt/'canonical-report.json'

def evaluate(limit):
    assert set(REPORTS)=={'musicfm','muq'}, 'Selesaikan kedua ekspor dahulu'
    reports=[json.loads(path.read_text()) for path in REPORTS.values()]
    assert all(r['candidate_for_quality_validation'] and r['smaller_than_previous_compiled'] for r in reports), 'Kandidat belum lebih kecil/ekspor gagal. Kirim canonical-report.json sebelum lanjut.'
    run_logged([RUN_PYTHON,'-u',ROOT/'scripts/canonical_encoder_quality.py',
        '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,'--reference-quality',QUALITY,'--graph-repair',REPAIR,
        '--palette-validation',PALETTE_VALIDATION,'--frozen-evaluation',FROZEN,'--baseline-pool-cache',BASELINE_POOL,
        '--musicfm-report',REPORTS['musicfm'],'--muq-report',REPORTS['muq'],
        '--output-dir',OUTPUT/'quality','--limit',limit],OUTPUT/('quality-one.log' if limit else 'quality-all.log'))


In [ ]:
export('musicfm')


In [ ]:
export('muq')


In [ ]:
from IPython.display import display,Markdown
lines=['| Encoder | Sebelum, compiled MiB | Kandidat, compiled MiB | Parameter short berubah | Parameter partial berubah |',
       '|---|---:|---:|---:|---:|']
for role,path in REPORTS.items():
    r=json.loads(path.read_text())
    changed=lambda profile:sum(v['changed'] for v in r['profiles'][profile]['differences'])
    lines.append(f"| {role} | {r['prior_compiled_bytes']/2**20:.1f} | {r['compiled_bytes']/2**20:.1f} | {changed('short_30')} | {changed('partial')} |")
    print(role,'report:',path)
display(Markdown('\n'.join(lines)))
print('Kualitas bobot short/partial belum disetujui. Uji satu lagu berikutnya.')


In [ ]:
evaluate(1)
smoke=json.loads((OUTPUT/'quality/validation-one.json').read_text())
print('Smoke metrics:',smoke['comparison_to_current_pipeline'])
print('Changed sections:',len(smoke['changed_sections']))


In [ ]:
# Smoke tidak mewakili full validation; bila ada penurunan, kirim report sebelum melanjutkan.
assert smoke['comparison_to_current_pipeline']['no_observed_drop'], 'Smoke turun. Kirim validation-one.json dan canonical-report.json; kandidat belum dipilih.'
evaluate(0)
r=json.loads((OUTPUT/'quality/validation-all.json').read_text())
assert r['full_validation'] and r['all_windows_reencoded'] and not r['partial_embeddings_reused'] and not r['test_used']
for label,key in [('Pipeline sebelumnya','baseline_current_pipeline'),('Kandidat satu bank','candidate')]:
    metric=r[key];print(label,'accuracy:',metric['section_accuracy'],'macro F1:',metric['macro_f1_present'])
print('Per-class delta:',json.dumps(r['comparison_to_current_pipeline'],indent=2))
print('Changed sections:',len(r['changed_sections']))
print('Routes:',r['routes'])
print('Kirim:',OUTPUT/'quality/validation-all.json')
print('Sertakan kedua canonical-report.json untuk ukuran dan kontrol parameter.')


Seluruh full dan partial encoder dijalankan ulang pada CPU Mac, sehingga tahap validasi ini memerlukan waktu. Hasil ringkas per lagu disimpan dan dapat dipakai ulang jika identitas sama; tidak membuat cache frame baru puluhan GiB. Jika interupsi, jalankan kembali cell yang sama. Setelah restart kernel, isi `RESUME_OUTPUT` dengan folder sebelumnya lalu jalankan setup/ekspor; hasil ekspor lengkap akan digunakan kembali.

Jika ekspor, guard parameter, kontrol native, smoke atau full quality gagal, jangan memasang kandidat ke aplikasi. Simpan report untuk diagnosis. Tidak menaikkan toleransi output atau mengubah anotasi agar hasil lolos. Jika ukuran dan kualitas memenuhi target, tahap selanjutnya adalah pengujian paket multi-function pada iPhone dengan frontend CPU dan encoder penuh CPU+GPU.
